# 03 · Ponto flutuante, Gauss e PLU

**Objetivo pedagógico:** Distinguir arredondamento, resíduo, erro e condicionamento e justificar o pivotamento.

**Luigi Scofano de Araujo · USP 13680334**

Reelaboração de estudo de um tema presente no acervo recuperado de 2025. Esta edição foi preparada em 2026. Não é uma cópia autenticada de uma entrega nem uma afirmação de conclusão das questões originais. Os arquivos recuperados estão em `acervo/`, com créditos e limitações em `docs/PROCEDENCIA.md`.

Execute as células em ordem, em kernel novo. Os dados são locais/sintéticos e as conclusões numéricas são verificadas por asserções; gráficos sozinhos não certificam convergência.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"numericos").is_dir() else Path.cwd().parent
if not (ROOT/"numericos").is_dir():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/.")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=7, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True})
FIGURAS = ROOT/"assets"/"figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
def mostrar(nome):
    plt.tight_layout()
    plt.savefig(FIGURAS/(nome+".svg"), bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close("all")

## 1. Representação e cancelamento
`float64` é uma aproximação binária. A subtração de números próximos pode perder dígitos significativos. Um teste de igualdade decimal não substitui uma análise de erro.

In [2]:
from numericos.lineares import fatorar_plu,resolver,residuo_relativo,gauss_reduzido
info=np.finfo(float)
print("epsilon:",info.eps,"maior float:",info.max)
print("(0.1+0.2)-0.3 =",(0.1+0.2)-0.3)
h=np.logspace(-16,-1,16)
direct=(np.sqrt(1+h)-1)/h
stable=1/(np.sqrt(1+h)+1)
plt.loglog(h,np.maximum(abs(direct-0.5),1e-18),"o-",label="subtração direta")
plt.loglog(h,np.maximum(abs(stable-0.5),1e-18),"s-",label="racionalizada")
plt.xlabel("h");plt.ylabel("erro frente ao limite 1/2");plt.legend()
mostrar("03_cancelamento")

epsilon: 2.220446049250313e-16 maior float: 1.7976931348623157e+308
(0.1+0.2)-0.3 = 5.551115123125783e-17


## 2. Pivotamento parcial e fatoração
A identidade usada é **P A = L U**. Trocar linhas em U exige trocar também as partes já construídas de L. A solução é obtida por `Ly=Pb`, seguida de `Ux=y`; as entradas não são alteradas.

In [3]:
A=np.array([[0.,2.,1.],[1.,-2.,-3.],[2.,3.,1.]])
x_exact=np.array([1.,-2.,3.]); b=A@x_exact
P,L,U=fatorar_plu(A)
x=resolver(A,b)
print("P=",P,"L=",L,"U=",U,sep="\n")
print("solução:",x,"resíduo normalizado:",residuo_relativo(A,x,b))
assert np.allclose(P@A,L@U,atol=1e-12)
assert np.allclose(x,x_exact,atol=1e-12)

P=
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
L=
[[ 1.         0.         0.       ]
 [ 0.5        1.         0.       ]
 [ 0.        -0.5714286  1.       ]]
U=
[[ 2.   3.   1. ]
 [ 0.  -3.5 -3.5]
 [ 0.   0.  -1. ]]
solução: [ 1. -2.  3.] resíduo normalizado: 0.0


## 3. Precisão decimal simulada
Este experimento arredonda entradas e operações para cinco algarismos significativos. Não simula exatamente um formato IEEE de 16 bits. A comparação ilustra uma matriz específica, não uma garantia universal de superioridade de erro direto em qualquer caso.

In [4]:
A=np.array([[1e-8,1.],[1.,1.]])
b=A@np.ones(2)
for pivot in [False,True]:
    x=gauss_reduzido(A,b,algarismos=5,pivotar=pivot)
    print("pivotamento:",pivot,"x:",x,"erro:",np.linalg.norm(x-1),"resíduo:",np.linalg.norm(A@x-b))
assert np.linalg.norm(gauss_reduzido(A,b,algarismos=5)-1)<1e-3

pivotamento: False x: [0. 1.] erro: 1.0 resíduo: 1.0
pivotamento: True x: [1. 1.] erro: 0.0 resíduo: 0.0


## 4. Condicionamento
Um resíduo pequeno não assegura um erro de solução pequeno quando A é mal condicionada. O exemplo de Hilbert testa uma solução conhecida; NumPy é uma referência adicional, não uma solução exata em aritmética real.

In [5]:
from scipy.linalg import hilbert
print("n    cond₂(A)      erro solução    resíduo normalizado")
for n in [3,6,9]:
    A=hilbert(n);b=A@np.ones(n);x=resolver(A,b)
    print(f"{n:2} {np.linalg.cond(A):12.3e} {np.linalg.norm(x-1,np.inf):14.3e} {residuo_relativo(A,x,b):18.3e}")
try:
    resolver([[1,2],[2,4]],[1,2])
except np.linalg.LinAlgError as error:
    print("Falha esperada:",error)
else:
    raise AssertionError("Sistema singular deveria ser rejeitado")

n    cond₂(A)      erro solução    resíduo normalizado
 3    5.241e+02      9.992e-16          0.000e+00
 6    1.495e+07      4.268e-10          9.063e-17
 9    4.932e+11      1.523e-05          3.924e-17
Falha esperada: Pivô numericamente nulo na escala de A.


## Para estudar
Explique cada troca de linha e reconstrua PA−LU. Compare uma perturbação em b com o erro em x. Identifique o limiar de pivô relativo à escala de A usado pelo módulo.

**Critério:** não chamar apenas o resíduo de “erro da solução”.

## Figuras desta execução

![03_cancelamento](../assets/figuras/03_cancelamento.svg)